# AIDP Race Intelligence: Parquet serving layer

This notebook publishes a read-only application projection from governed Gold Delta tables to OCI Object Storage Volume paths. Delta Gold remains the governed source of truth. The Parquet files are external serving projections only: Autonomous AI Lakehouse can read them without copying them into database storage.

## Load governed Gold inputs

Run after notebooks 00–07. This notebook does not change Gold tables, modelling methodology, or the historical/synthetic boundary.

In [ ]:
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession, Window, functions as F

spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')

sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import load_aidp_config, table_name

config = load_aidp_config(root / 'config/settings.yaml')

In [ ]:
serving_root = '/Volumes/race_intelligence/gold/serving'
exports = {
    'observed_driver_profiles': {
        'source': table_name(config, 'gold_historical_schema', 'observed_driver_profiles'),
        'target': f'{serving_root}/gold_historical/observed_driver_profiles',
        'critical_columns': {'driver_id', 'pooled_median_pace', 'support_status'},
    },
    'historical_variability': {
        'source': table_name(config, 'gold_historical_schema', 'historical_variability'),
        'target': f'{serving_root}/gold_historical/historical_variability',
        'critical_columns': {'session', 'driver_id', 'lap_number', 'lap_deviation'},
    },
    'virtual_max_profile': {
        'source': table_name(config, 'gold_scenario_schema', 'virtual_max_profile'),
        'target': f'{serving_root}/gold_scenario/virtual_max_profile',
        'critical_columns': {'scenario_driver_id', 'virtual_max_pace_seconds', 'scenario_type', 'label'},
    },
    'monte_carlo_scenario_summary': {
        'source': table_name(config, 'gold_scenario_schema', 'monte_carlo_scenario_summary'),
        'target': f'{serving_root}/gold_scenario/monte_carlo_scenario_summary',
        'critical_columns': {'supported_field_size', 'race_laps', 'simulations', 'probability_at_zero_delay'},
    },
    'monte_carlo_probability_curve': {
        'source': table_name(config, 'gold_scenario_schema', 'monte_carlo_probability_curve'),
        'target': f'{serving_root}/gold_scenario/monte_carlo_probability_curve',
        'critical_columns': {'delay_seconds', 'full_field_catch_probability'},
    },
}

missing_sources = [name for name, item in exports.items() if not spark.catalog.tableExists(item['source'])]
if missing_sources:
    raise RuntimeError(f'Missing required governed Gold source table(s): {", ".join(missing_sources)}')

source_frames = {name: spark.table(item['source']) for name, item in exports.items()}
for name, item in exports.items():
    missing_columns = item['critical_columns'] - set(source_frames[name].columns)
    if missing_columns:
        raise RuntimeError(f'{name} is missing critical Gold column(s): {", ".join(sorted(missing_columns))}')

print('All governed Gold source tables and critical columns are available.')

## Publish external Parquet projections

Each small serving dataset is coalesced to one Parquet output partition and overwritten atomically at its serving path. No Parquet export is registered as a Gold table.

In [ ]:
for name, item in exports.items():
    source_frames[name].coalesce(1).write.mode('overwrite').parquet(item['target'])
    print(f'Published serving projection: {name}')

## Read-back validation

Validation is performed against the Parquet projections, not against a cached source DataFrame. Historical and synthetic scenario data remain separate paths.

In [ ]:
serving_frames = {name: spark.read.parquet(item['target']) for name, item in exports.items()}
manifest_rows = []

for name, item in exports.items():
    frame = serving_frames[name]
    row_count = frame.count()
    missing_columns = item['critical_columns'] - set(frame.columns)
    if row_count <= 0:
        raise RuntimeError(f'Serving projection {name} has no rows after write.')
    if missing_columns:
        raise RuntimeError(f'Serving projection {name} is missing critical column(s): {", ".join(sorted(missing_columns))}')
    manifest_rows.append((name, item['source'], item['target'], int(row_count)))

profiles = serving_frames['observed_driver_profiles']
variability = serving_frames['historical_variability']
virtual_max = serving_frames['virtual_max_profile']
summary = serving_frames['monte_carlo_scenario_summary']
curve = serving_frames['monte_carlo_probability_curve']

supported_profiles = profiles.filter(F.col('support_status') == 'sufficient').count()
if profiles.filter(F.col('driver_id').cast('string').rlike('VIRTUAL_MAX|SYN_')).count() != 0:
    raise ValueError('Historical serving projection contains synthetic driver records.')
if variability.filter(F.col('lap_deviation').isNull()).count() != 0:
    raise ValueError('Historical variability serving projection contains null deviations.')
if variability.filter(F.col('driver_id').cast('string').rlike('VIRTUAL_MAX|SYN_')).count() != 0:
    raise ValueError('Historical variability serving projection contains synthetic records.')
if virtual_max.filter("scenario_type = 'SYNTHETIC' AND label = 'WHAT-IF'").count() != 1:
    raise ValueError('Virtual Max serving projection must remain one synthetic WHAT-IF profile.')

summary_row = summary.orderBy(F.col('simulations').desc()).first()
if summary_row is None or int(summary_row.supported_field_size) != supported_profiles:
    raise ValueError('Scenario summary supported field size does not match observed supported profiles.')
if not 0.0 <= float(summary_row.probability_at_zero_delay) <= 1.0:
    raise ValueError('Scenario summary probability_at_zero_delay must be between 0 and 1.')

curve_window = Window.orderBy('delay_seconds')
curve_with_previous = curve.withColumn('_previous_probability', F.lag('full_field_catch_probability').over(curve_window))
if curve.filter((F.col('full_field_catch_probability') < 0) | (F.col('full_field_catch_probability') > 1)).count() != 0:
    raise ValueError('Probability curve values must be between 0 and 1.')
if curve.groupBy('delay_seconds').count().filter('count > 1').count() != 0:
    raise ValueError('Probability curve delay values must be unique for serving use.')
if curve_with_previous.filter(F.col('full_field_catch_probability') > F.col('_previous_probability')).count() != 0:
    raise ValueError('Probability curve must be non-increasing as starting delay increases.')

print('Serving read-back validation passed.')

## Serving manifest

The resulting Parquet files are application-facing projections only. Delta Gold remains the governed source of truth; Autonomous AI Lakehouse reads these external paths without duplicating the data into database storage.

In [ ]:
serving_manifest = spark.createDataFrame(
    manifest_rows,
    'dataset string, source_gold_table string, target_serving_path string, row_count long',
).orderBy('dataset')
serving_manifest.show(truncate=False)
print('Historical projection: gold_historical/observed_driver_profiles')
print('Synthetic scenario projections: gold_scenario/*')